# Tracking Evaluation (Phase 4)

Runs in Google Colab. Any dataset-scale tracking evaluation happens here,
not on a laptop. This notebook calls `evat.tracking.*` / `evat.video.*` /
`evat.visualization.tracking_overlay` — it does not reimplement matching
or lifecycle logic.

Dataset: YouTube-VOS (non-commercial research use only). DAVIS is not
used; MOT17 is not substituted.

This is a baseline (mask-IoU greedy matching) tracker evaluation, not a
production-grade or benchmark-comparable (MOTA/HOTA/IDF1) result.

In [ ]:
%pip install -q -e .

In [ ]:
import os
from pathlib import Path

os.environ["EVAT_DATA_ROOT"] = "/content/data"  # example only; set to the real path
dataset_root = Path(os.environ["EVAT_DATA_ROOT"]) / "youtube_vos"

## SMOKE EXPERIMENT

Verify frames load, masks load, object IDs load, the tracker produces
tracks, visualization works, and metrics calculate correctly — on a tiny
subset — before any larger evaluation.

In [ ]:
from evat.data.datasets.youtube_vos import build_video_index
from evat.tracking.ground_truth import extract_ground_truth_instances, strip_identity
from evat.tracking.metrics import evaluate_tracking
from evat.tracking.tracker import Tracker, TrackerConfig
from evat.video.sampling import uniform_frame_indices
from evat.video.sequence import build_temporal_sequence
from evat.video.tensors import load_temporal_sequence
from evat.visualization.tracking_overlay import draw_tracks

config = TrackerConfig.from_yaml("configs/tracking.yaml")

all_videos = build_video_index(dataset_root, split="train")
smoke_videos = all_videos[:2]


def run_tracking_on_video(video, num_samples=8):
    indices = uniform_frame_indices(num_frames_total=len(video.frames), num_samples=num_samples)
    sequence = build_temporal_sequence(video, indices)
    batch = load_temporal_sequence(sequence, dataset_root=dataset_root)

    tracker = Tracker(config)
    gt_by_frame, predictions_by_frame = {}, {}
    for frame_id, object_id_mask in zip(batch.frame_ids, batch.masks, strict=True):
        if object_id_mask is None:
            gt_by_frame[frame_id] = []
            predictions_by_frame[frame_id] = tracker.update(frame_id, [])
            continue
        gt_instances = extract_ground_truth_instances(object_id_mask, frame_id)
        gt_by_frame[frame_id] = gt_instances
        predictions_by_frame[frame_id] = tracker.update(frame_id, strip_identity(gt_instances))

    return batch, gt_by_frame, predictions_by_frame


for video in smoke_videos:
    batch, gt_by_frame, predictions_by_frame = run_tracking_on_video(video)
    metrics = evaluate_tracking(gt_by_frame, predictions_by_frame, list(batch.frame_ids))
    print(video.video_id, metrics)

## BASELINE EVALUATION

Only run after the smoke experiment succeeds. Uses the official YouTube-VOS
validation split — never used for tuning, only for reporting.

In [ ]:
import time

val_videos = build_video_index(dataset_root, split="valid")

start = time.time()
total_frames = 0
all_metrics = []
for video in val_videos:
    batch, gt_by_frame, predictions_by_frame = run_tracking_on_video(video)
    total_frames += len(batch.frame_ids)
    all_metrics.append(evaluate_tracking(gt_by_frame, predictions_by_frame, list(batch.frame_ids)))
runtime_seconds = time.time() - start

print("videos:", len(val_videos))
print("frames processed:", total_frames)
print("runtime_seconds:", runtime_seconds)
print("approx FPS:", total_frames / runtime_seconds if runtime_seconds > 0 else float("nan"))
print("mean coverage:", sum(m.coverage for m in all_metrics) / len(all_metrics))
print("mean id_consistency:", sum(m.id_consistency for m in all_metrics) / len(all_metrics))
print("total identity_switches:", sum(m.identity_switches for m in all_metrics))

In [ ]:
# Qualitative inspection: draw tracks on a sample sequence.
import numpy as np

sample_video = val_videos[0]
batch, gt_by_frame, predictions_by_frame = run_tracking_on_video(sample_video)

for i, frame_id in enumerate(batch.frame_ids[:3]):
    frame_rgb = np.transpose(batch.images[i], (1, 2, 0))
    panel = draw_tracks(frame_rgb, predictions_by_frame[frame_id])
    panel.save(f"results/tracking/sample_{sample_video.video_id}_{frame_id}.png")

## Save results

Record the actual printed metrics/runtime above into `docs/experiments.md`.
Do not hand-edit numbers that were not produced by this notebook. Do not
call this "real-time" unless the measured FPS and methodology justify it.